## 1. Load Libraries and Configure Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries loaded successfully")

## 2. Data Loading Functions

Parse LETOR format: `relevance qid:query_id 1:feat 2:feat ... 25:feat #docid=id`

In [ ]:
def load_letor_data(filepath):
    """
    Load LETOR format data.
    Format: relevance qid:query_id 1:feature ... 25:feature #docid=doc_id
    
    Returns:
        X: (n_samples, 25) feature matrix
        y: (n_samples,) relevance labels
        qid: (n_samples,) query IDs for grouping
        docids: (n_samples,) document IDs
    """
    X = []
    y = []
    qid = []
    docids = []
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Parse line
            parts = line.split('#')[0].strip().split()
            
            # Extract relevance
            relevance = int(parts[0])
            y.append(relevance)
            
            # Extract qid
            qid_str = parts[1].split(':')[1]
            qid.append(int(qid_str))
            
            # Extract features
            features = np.zeros(25)
            for feat_part in parts[2:]:
                feat_idx, feat_val = feat_part.split(':')
                features[int(feat_idx) - 1] = float(feat_val)
            X.append(features)
            
            # Extract docid
            docid_part = line.split('#docid = ')[-1].strip()
            docid = int(docid_part) if docid_part else -1
            docids.append(docid)
    
    return np.array(X), np.array(y), np.array(qid), np.array(docids)

# Test on Fold1 training data (path: repo root or partB/)
DATA_ROOT = Path('data/letor/OHSUMED/Data')
if not (DATA_ROOT / 'Fold1').exists():
    DATA_ROOT = Path('../data/letor/OHSUMED/Data')  # when run from partB/
if not (DATA_ROOT / 'Fold1').exists():
    DATA_ROOT = Path('data/letor/OHSUMED/Data')  # fallback
data_path = DATA_ROOT / 'Fold1'
X_train, y_train, qid_train, docid_train = load_letor_data(data_path / 'trainingset.txt')

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Unique queries: {len(np.unique(qid_train))}")
print(f"Relevance distribution:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"\nFeature stats:")
print(f"  Min: {X_train.min():.4f}")
print(f"  Max: {X_train.max():.4f}")
print(f"  Mean: {X_train.mean():.4f}")

## 3. Load All Folds for Cross-Validation

In [ ]:
# Load all 5 folds
folds = {}
for fold_idx in range(1, 6):
    fold_path = DATA_ROOT / f'Fold{fold_idx}'
    
    X_train, y_train, qid_train, _ = load_letor_data(fold_path / 'trainingset.txt')
    X_val, y_val, qid_val, _ = load_letor_data(fold_path / 'validationset.txt')
    X_test, y_test, qid_test, _ = load_letor_data(fold_path / 'testset.txt')
    
    folds[fold_idx] = {
        'X_train': X_train, 'y_train': y_train, 'qid_train': qid_train,
        'X_val': X_val, 'y_val': y_val, 'qid_val': qid_val,
        'X_test': X_test, 'y_test': y_test, 'qid_test': qid_test
    }
    
    print(f"Fold {fold_idx}:")
    print(f"  Train: {X_train.shape[0]} samples, {len(np.unique(qid_train))} queries")
    print(f"  Val:   {X_val.shape[0]} samples, {len(np.unique(qid_val))} queries")
    print(f"  Test:  {X_test.shape[0]} samples, {len(np.unique(qid_test))} queries")

## 4. Normalize Features

In [ ]:
from sklearn.preprocessing import StandardScaler

def normalize_fold(fold_data):
    """
    Normalize features using StandardScaler fitted on training set.
    """
    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(fold_data['X_train'])
    X_val_norm = scaler.transform(fold_data['X_val'])
    X_test_norm = scaler.transform(fold_data['X_test'])
    
    return {
        'X_train': X_train_norm, 'y_train': fold_data['y_train'], 'qid_train': fold_data['qid_train'],
        'X_val': X_val_norm, 'y_val': fold_data['y_val'], 'qid_val': fold_data['qid_val'],
        'X_test': X_test_norm, 'y_test': fold_data['y_test'], 'qid_test': fold_data['qid_test']
    }

# Normalize all folds
for fold_idx in range(1, 6):
    folds[fold_idx] = normalize_fold(folds[fold_idx])

print("Features normalized successfully")